In [1]:
pip install langchain huggingface_hub langchain_community tiktoken rank_bm25 pypdf lancedb langchain-groq


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")
if groq_api_key:
    os.environ["GROQ_API_KEY"] = groq_api_key

BM25 Retriever - Sparse retriever

Embeddings - Dense retrievers Lancedb

In [3]:
import requests
import os

# Download a public PDF (Attention is All You Need paper from arXiv)
pdf_url = "https://arxiv.org/pdf/1706.03762.pdf"
pdf_path = "sample.pdf"

response = requests.get(pdf_url)
with open(pdf_path, 'wb') as f:
    f.write(response.content)
print(f"Downloaded PDF to {pdf_path}")

Downloaded PDF to sample.pdf


In [4]:
# Load the PDF
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("sample.pdf")
pages = loader.load_and_split()
print(f"Loaded {len(pages)} pages from the PDF")

/var/folders/0m/db48z_8d0m5gfh8zkg4vrk100000gn/T/ipykernel_1642/3600661614.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/Users/dhruvdiwakirti/Who-killed-Rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 16 pages from the PDF


In [5]:
from langchain_community.vectorstores import LanceDB
import lancedb
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.documents import Document
from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_classic.chains import RetrievalQA


In [6]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/var/folders/0m/db48z_8d0m5gfh8zkg4vrk100000gn/T/ipykernel_1642/1154258705.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9793.78it/s]


In [7]:
# Initialize the BM25 retriever
bm25_retriever = BM25Retriever.from_documents(pages)
bm25_retriever.k = 2  # Retrieve top 2 results

print("type of bm25", type(bm25_retriever))

type of bm25 <class 'langchain_community.retrievers.bm25.BM25Retriever'>


In [8]:
db = lancedb.connect("/tmp/lancedb")
table = db.create_table(
    "pandas_docs",
    data=[
        {
            "vector": embedding.embed_query("Hello World"),
            "text": "Hello World",
            "id": "1",
        }
    ],
    mode="overwrite",
)

[2026-06-24T18:09:09Z WARN  lance::dataset::write::insert] No existing dataset at /tmp/lancedb/pandas_docs.lance, it will be created


In [9]:
# Initialize LanceDB retriever
docsearch = LanceDB.from_documents(pages, embedding, connection=db)
retriever_lancedb = docsearch.as_retriever(search_kwargs={"k": 2})

# Initialize the ensemble retriever
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, retriever_lancedb], weights=[0.2, 0.8]
)

In [10]:
# Example customer query
query = "what is cross attention?"


# Retrieve relevant documents/products
docs = ensemble_retriever.invoke(query)

# Extract and print only the page content from each document
# for doc in docs:
#     print(doc.page_content)

docs

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'sample.pdf', 'total_pages': 15, 'page': 12, 'page_label': '13'}, page_content='Attention Visualizations\nInput-Input Layer5\nIt\nis\nin\nthis\nspirit\nthat\na\nmajority\nof\nAmerican\ngovernments\nhave\npassed\nnew\nlaws\nsince\n2009\nmaking\nthe\nregistration\nor\nvoting\nprocess\nmore\ndifficult\n.\n<EOS>\n<pad>\n<pad>\n<pad>\n<pad>\n<pad>\n<pad>\nIt\nis\nin\nthis\nspirit\nthat\na\nmajority\nof\nAmerican\ngovernments\nhave\npassed\nnew\nlaws\nsince\n2009\nmaking\nthe\nregistration\nor\nvoting\nprocess\nmore\ndifficult\n.\n<EOS>\n<pad>\n<pad>\n<pad>\n<pad>\n<pad>\n<pad>\nFigure 3: An example of the attention mechanism follo

## RAGAS Evaluation of the Hybrid RAG

Score the BM25 + LanceDB ensemble retriever (with a Groq LLM for generation) using RAGAS:

- **context_precision** & **context_recall** — quality of what the hybrid retriever pulls back
- **faithfulness** — is the generated answer grounded in the retrieved context?
- **answer_relevancy** — does the answer actually address the question?

Groq (`llama-3.3-70b-versatile`) acts as the judge LLM; the same local HuggingFace embeddings power `answer_relevancy`. No OpenAI key required.

In [11]:
# Already present in this env; uncomment to install elsewhere.
# %pip install -U ragas datasets

In [12]:
# NOTE: ragas 0.4.3 imports `langchain_community.chat_models.vertexai`, which was
# removed in the installed langchain-community. We stub that module so ragas imports
# cleanly (the stub is only touched if you explicitly select a Vertex AI model).
import sys, types

if "langchain_community.chat_models.vertexai" not in sys.modules:
    _stub = types.ModuleType("langchain_community.chat_models.vertexai")
    _stub.ChatVertexAI = type("ChatVertexAI", (), {})
    sys.modules["langchain_community.chat_models.vertexai"] = _stub

from ragas import evaluate, EvaluationDataset
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig

/var/folders/0m/db48z_8d0m5gfh8zkg4vrk100000gn/T/ipykernel_1642/3051984823.py:12: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/var/folders/0m/db48z_8d0m5gfh8zkg4vrk100000gn/T/ipykernel_1642/3051984823.py:12: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
/var/folders/0m/db48z_8d0m5gfh8zkg4vrk100000gn/T/ipykernel_1642/3051984823.py:12: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
/

In [13]:
# Generation LLM (Groq) + a simple RAG chain on top of the hybrid retriever
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

rag_prompt = ChatPromptTemplate.from_template(
    "Answer the question using ONLY the context below. "
    "If the answer is not in the context, say you don't know.\n\n"
    "Context:\n{context}\n\nQuestion: {question}\nAnswer:"
)

def rag_answer(question: str):
    """Run the hybrid retriever + LLM. Returns (answer, list_of_context_strings)."""
    docs = ensemble_retriever.invoke(question)
    contexts = [d.page_content for d in docs]
    context_text = "\n\n".join(contexts)
    messages = rag_prompt.format_messages(context=context_text, question=question)
    answer = llm.invoke(messages).content
    return answer, contexts

In [14]:
# Evaluation set: questions + reference (ground-truth) answers grounded in the paper
eval_data = [
    {
        "question": "How many layers (N) does the Transformer encoder have?",
        "ground_truth": "The encoder is composed of a stack of N = 6 identical layers.",
    },
    {
        "question": "What is the dimensionality of the model, d_model?",
        "ground_truth": "The model dimensionality d_model is 512.",
    },
    {
        "question": "What activation function is used in the position-wise feed-forward networks?",
        "ground_truth": "A ReLU activation is used between the two linear transformations.",
    },
    {
        "question": "How many attention heads does the model use, and what is the dimension per head?",
        "ground_truth": "The model uses h = 8 parallel attention heads, each with dimension d_k = d_v = d_model / h = 64.",
    },
    {
        "question": "What is multi-head attention?",
        "ground_truth": (
            "Multi-head attention lets the model jointly attend to information from different "
            "representation subspaces at different positions, by running several attention "
            "functions in parallel and concatenating and projecting their outputs."
        ),
    },
]

In [15]:
# Run the hybrid RAG over every question to collect responses + retrieved contexts
samples = []
for row in eval_data:
    answer, contexts = rag_answer(row["question"])
    samples.append({
        "user_input": row["question"],
        "retrieved_contexts": contexts,
        "response": answer,
        "reference": row["ground_truth"],
    })

evaluation_dataset = EvaluationDataset.from_list(samples)
evaluation_dataset.to_pandas()

,user_input,retrieved_contexts,response,reference
0,How many layers (N) does the Transformer encod...,[Figure 1: The Transformer - model architectur...,The Transformer encoder has **N = 6 layers**.,The encoder is composed of a stack of N = 6 id...
1,"What is the dimensionality of the model, d_model?",[Table 3: Variations on the Transformer archit...,The model’s dimensionality \(d_{\text{model}}\...,The model dimensionality d_model is 512.
2,What activation function is used in the positi...,[Figure 1: The Transformer - model architectur...,ReLU.,A ReLU activation is used between the two line...
3,"How many attention heads does the model use, a...",[Input-Input Layer5\nThe\nLaw\nwill\nnever\nbe...,"The model uses **8 attention heads**, and each...","The model uses h = 8 parallel attention heads,..."
4,What is multi-head attention?,[output values. These are concatenated and onc...,Multi‑head attention is an attention mechanism...,Multi-head attention lets the model jointly at...


In [16]:
# Wrap Groq as the judge LLM and reuse the HF embeddings (for answer_relevancy),
# then score the hybrid RAG with the core RAGAS metrics.
evaluator_llm = LangchainLLMWrapper(llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(embedding)

# answer_relevancy normally asks the judge for `strictness` completions in one call
# (n=3). Groq only allows n=1, so we set strictness=1 to avoid a 400 error.
answer_relevancy.strictness = 1

result = evaluate(
    dataset=evaluation_dataset,
    metrics=[
        context_precision,   # are retrieved chunks relevant to the question?
        context_recall,      # did retrieval cover the reference answer?
        faithfulness,        # is the answer grounded in the retrieved context?
        answer_relevancy,    # does the answer actually address the question?
    ],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    run_config=RunConfig(max_workers=2),  # keep low to respect Groq rate limits
)
result

/var/folders/0m/db48z_8d0m5gfh8zkg4vrk100000gn/T/ipykernel_1642/1491913082.py:3: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(llm)
/var/folders/0m/db48z_8d0m5gfh8zkg4vrk100000gn/T/ipykernel_1642/1491913082.py:4: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  evaluator_embeddings = LangchainEmbeddingsWrapper(embedding)
Evaluating: 100%|██████████| 20/20 [08:30<00:00, 25.54s/it]


{'context_precision': 0.7278, 'context_recall': 1.0000, 'faithfulness': 1.0000, 'answer_relevancy': 0.8764}

In [17]:
# Per-sample scores (one row per question)
result.to_pandas()

,user_input,retrieved_contexts,response,reference,context_precision,context_recall,faithfulness,answer_relevancy
0,How many layers (N) does the Transformer encod...,[Figure 1: The Transformer - model architectur...,The Transformer encoder has **N = 6 layers**.,The encoder is composed of a stack of N = 6 id...,1.000000,1.0,1.0,0.974440
1,"What is the dimensionality of the model, d_model?",[Table 3: Variations on the Transformer archit...,The model’s dimensionality \(d_{\text{model}}\...,The model dimensionality d_model is 512.,0.805556,NaN,1.0,0.974972
2,What activation function is used in the positi...,[Figure 1: The Transformer - model architectur...,ReLU.,A ReLU activation is used between the two line...,0.333333,1.0,1.0,0.473116
3,"How many attention heads does the model use, a...",[Input-Input Layer5\nThe\nLaw\nwill\nnever\nbe...,"The model uses **8 attention heads**, and each...","The model uses h = 8 parallel attention heads,...",0.500000,1.0,1.0,0.980231
4,What is multi-head attention?,[output values. These are concatenated and onc...,Multi‑head attention is an attention mechanism...,Multi-head attention lets the model jointly at...,1.000000,1.0,1.0,0.979156


## Adding a reranker

The hybrid (BM25 + LanceDB) retriever is good at *recall* — it pulls back a pool of
plausible chunks — but its ordering is coarse. A **reranker** is a precise second stage
that re-scores each candidate against the query and keeps only the best few. The pattern
is **retrieve-then-rerank**: over-retrieve cheaply, then reorder carefully.

We use the [`rerankers`](https://github.com/AnswerDotAI/rerankers) library, which gives one
unified API across very different reranker families, so swapping models is a single line:

1. **Cross-encoder** — the classic BERT-style baseline.
2. **ColBERT** — late interaction (token-level matching).
3. **LLM reranker** — listwise ordering by an LLM (RankGPT, via the existing Groq model).

In [18]:
# Install rerankers. NOTE: run this BEFORE the LanceDB cells (or from a terminal),
# because %pip forks a subprocess and an already-open LanceDB connection doesn't
# survive the fork -- you'll see a harmless "Bad file descriptor" traceback if so.
# This guard makes re-running a no-op once it's installed.
try:
    import rerankers  # noqa: F401
    print("rerankers already installed:", rerankers.__version__)
except ImportError:
    %pip install -q "rerankers[transformers]"
    print("Installed rerankers -- restart the kernel before using it.")

rerankers already installed: 0.10.0


### Step 1: Widen the candidate pool + a rerank helper

A reranker can only reorder what it's handed, so we bump the retrievers from `k=2` to
`k=8` to give it a real pool, then keep the best 3 after reranking. The `rerank()` helper
is generic — every step below just passes a different `ranker` into it.

In [19]:
# Over-retrieve so the reranker has a real pool of candidates to reorder.
bm25_retriever.k = 8
retriever_lancedb = docsearch.as_retriever(search_kwargs={"k": 8})
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, retriever_lancedb], weights=[0.2, 0.8]
)


def rerank(ranker, query, top_k=3):
    """Hybrid-retrieve a wide pool, then reorder it with `ranker`; keep the top_k."""
    docs = ensemble_retriever.invoke(query)
    texts = [d.page_content for d in docs]
    ranked = ranker.rank(query=query, docs=texts)
    return [(r.score, r.text) for r in ranked.top_k(top_k)]


def show(results):
    for score, text in results:
        s = f"{score:.3f}" if score is not None else "  -  "
        print(f"{s}  {text[:120].strip()!r}")


query = "what is transformer model?"

### Step 2: Run a cross-encoder (the baseline reranker)

A **cross-encoder** feeds the query and a candidate document through the model *together*
and outputs a single relevance score. Because it sees both at once, it's far more accurate
than the bi-encoder used for retrieval — but it has to run once per candidate, so it's
only practical as a second stage on a small pool. This is the standard reranking baseline.

In [20]:
from rerankers import Reranker

# Small, fast MS MARCO cross-encoder as the baseline reranker.
ce_ranker = Reranker("cross-encoder/ms-marco-MiniLM-L-6-v2", model_type="cross-encoder")
show(rerank(ce_ranker, query))

Loading TransformerRanker model cross-encoder/ms-marco-MiniLM-L-6-v2 (this message can be suppressed by setting verbose=0)
No device set
Using device mps
No dtype set
Using dtype torch.float32


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 11186.52it/s]


Loaded model cross-encoder/ms-marco-MiniLM-L-6-v2
Using device mps.
Using dtype torch.float32.
3.348  'Figure 1: The Transformer - model architecture.\nThe Transformer follows this overall architecture using stacked self-att'
1.087  'Table 4: The Transformer generalizes well to English constituency parsing (Results are on Section 23\nof WSJ)\nParser Trai'
0.210  'Table 2: The Transformer achieves better BLEU scores than previous state-of-the-art models on the\nEnglish-to-German and'


### Step 3: Swap to ColBERT (late interaction)

ColBERT embeds **every** query token and **every** document token, then scores via
**MaxSim** — each query token is matched to its best-matching document token and the
similarities are summed. This token-level ("late interaction") matching captures more
nuance than a single pooled vector, while staying cheaper than a full cross-encoder.
Thanks to the unified API, swapping it in is a one-line change.

In [21]:
# Compatibility shim: rerankers 0.10.0's ColBERTModel calls the old `init_weights()`,
# but transformers 5.x only sets up `all_tied_weights_keys` (and friends) inside
# `post_init()`. Without this, Reranker(..., model_type="colbert") raises
# AttributeError: 'ColBERTModel' object has no attribute 'all_tied_weights_keys'.
from rerankers.models import colbert_ranker

if not getattr(colbert_ranker.ColBERTModel, "_post_init_patched", False):
    _orig_colbert_init = colbert_ranker.ColBERTModel.__init__

    def _patched_colbert_init(self, config, verbose):
        _orig_colbert_init(self, config, verbose)
        if not hasattr(self, "all_tied_weights_keys"):
            self.post_init()

    colbert_ranker.ColBERTModel.__init__ = _patched_colbert_init
    colbert_ranker.ColBERTModel._post_init_patched = True

colbert_reranker = Reranker("colbert-ir/colbertv2.0", model_type="colbert")
show(rerank(colbert_reranker, query))

Loading ColBERTRanker model colbert-ir/colbertv2.0 (this message can be suppressed by setting verbose=0)
No device set
Using device mps
No dtype set
Using dtype torch.float32
Loading model colbert-ir/colbertv2.0, this might take a while...
Linear Dim set to: 128 for downcasting


Loading weights: 100%|██████████| 200/200 [00:00<00:00, 7857.59it/s]


1.117  'Figure 1: The Transformer - model architecture.\nThe Transformer follows this overall architecture using stacked self-att'
1.045  'Table 4: The Transformer generalizes well to English constituency parsing (Results are on Section 23\nof WSJ)\nParser Trai'
1.025  'Table 2: The Transformer achieves better BLEU scores than previous state-of-the-art models on the\nEnglish-to-German and'


### Step 4: LLM reranker (listwise)

An LLM reranker (**RankGPT**) hands the *whole* candidate list to an LLM and asks it to
order the documents by relevance — a listwise decision rather than scoring each doc in
isolation. It's the most capable and the most expensive of the three. `rerankers` routes
the call through LiteLLM, so we reuse the same Groq model configured earlier via the
`groq/` model prefix (no new API key needed).

In [22]:
llm_ranker = Reranker(
    "groq/openai/gpt-oss-120b",
    model_type="rankgpt",
    api_key=os.environ["GROQ_API_KEY"],
)
show(rerank(llm_ranker, query))

Loading RankGPTRanker model groq/openai/gpt-oss-120b (this message can be suppressed by setting verbose=0)
Querying model groq/openai/gpt-oss-120b with via LiteLLM...
  -    'Figure 1: The Transformer - model architecture.\nThe Transformer follows this overall architecture using stacked self-att'
  -    'Table 2: The Transformer achieves better BLEU scores than previous state-of-the-art models on the\nEnglish-to-German and'
  -    'Table 3: Variations on the Transformer architecture. Unlisted values are identical to those of the base\nmodel. All metri'


## Does reranking help? Baseline vs cross-encoder (RAGAS A/B)

We've reordered candidate lists — but does that translate into a better RAG? Here we hold
everything constant (same questions, same wide `k=8` pool, same generation LLM) and change
**only the final selection**:

- **baseline** — keep the top 3 by the hybrid fusion score (no reranker)
- **cross-encoder rerank** — keep the top 3 after the cross-encoder rescores the same pool

We score the two retrieval-quality metrics reranking targets:

- **context_precision** — should *rise* as off-topic chunks get pushed out of the top 3
- **context_recall** — should *hold* (we don't want reranking to drop the chunk with the answer)

The cross-encoder runs locally (free); only the RAGAS judge uses Groq. If you hit a 429,
the daily token limit is exhausted — shrink `subset` or wait for the reset.

In [23]:
import pandas as pd
from ragas.metrics import context_precision, context_recall

# The two retrieval-quality metrics reranking targets. We drop faithfulness /
# answer_relevancy here to roughly halve the judge calls (Groq quota); add them
# back to this list once your daily token limit has reset.
ab_metrics = [context_precision, context_recall]
metric_names = [m.name for m in ab_metrics]

# Subset of questions to stay under the Groq tokens-per-day limit.
# Bump to `eval_data` for the full set once quota allows.
subset = eval_data


def build_dataset(ranker=None, top_k=3):
    """Retrieve the wide (k=8) pool, then keep top_k either by hybrid score
    (ranker=None) or after reranking; generate an answer and package for RAGAS."""
    samples = []
    for row in subset:
        q = row["question"]
        docs = ensemble_retriever.invoke(q)              # wide pool (k=8)
        texts = [d.page_content for d in docs]
        if ranker is None:
            texts = texts[:top_k]                        # top_k by hybrid fusion
        else:
            texts = [t for _, t in rerank(ranker, q, top_k=top_k)]
        context_text = "\n\n".join(texts)
        messages = rag_prompt.format_messages(context=context_text, question=q)
        answer = llm.invoke(messages).content
        samples.append({
            "user_input": q,
            "retrieved_contexts": texts,
            "response": answer,
            "reference": row["ground_truth"],
        })
    return EvaluationDataset.from_list(samples)


cfg = RunConfig(max_workers=1)  # gentle on Groq rate limits

baseline_eval = evaluate(
    dataset=build_dataset(ranker=None),
    metrics=ab_metrics, llm=evaluator_llm,
    embeddings=evaluator_embeddings, run_config=cfg,
)
reranked_eval = evaluate(
    dataset=build_dataset(ranker=ce_ranker),
    metrics=ab_metrics, llm=evaluator_llm,
    embeddings=evaluator_embeddings, run_config=cfg,
)


def agg(ev):
    df = ev.to_pandas()
    return {n: df[n].mean() for n in metric_names}


comparison = pd.DataFrame({
    "baseline (top-3 hybrid)": agg(baseline_eval),
    "cross-encoder rerank": agg(reranked_eval),
})
comparison["delta"] = comparison["cross-encoder rerank"] - comparison["baseline (top-3 hybrid)"]
comparison.round(4)

/var/folders/0m/db48z_8d0m5gfh8zkg4vrk100000gn/T/ipykernel_1642/2026528618.py:2: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import context_precision, context_recall
/var/folders/0m/db48z_8d0m5gfh8zkg4vrk100000gn/T/ipykernel_1642/2026528618.py:2: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import context_precision, context_recall


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01k5hjn0e6fyxbf0wfmz4aha4n` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 197724, Requested 2427. Please try again in 1m5.232s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

# Making the evaluation trustworthy (3 fixes)

The earlier evals *looked* great (recall / faithfulness = 1.0) but weren't reliable:
only **5 easy factoid questions**, the **judge model was the generator model**, and
**page-level chunks** that cap precision. The cells below fix exactly those, then re-run
the A/B as a proper reranker sweep:

1. **Bigger eval set** — auto-generate diverse Q/A pairs from the paper with RAGAS.
2. **Smaller chunks** — re-split the PDF so retrieval & reranking work on cleaner units.
3. **Separate judge** — grade with a *different* Groq model than the one writing answers.

**Run order:** execute the original notebook top-to-bottom first (so `pages`, `embedding`,
`db`, `llm`, `rag_prompt`, `ce_ranker`, `colbert_reranker`, `llm_ranker`,
`evaluator_embeddings` all exist), then run these cells in order.

> ⚠️ **Groq quota.** These cells make many calls (synthetic generation + 4 RAG configs ×
> N questions × 4 judge metrics). On a 429 / daily-token limit, lower `N_SYNTHETIC`,
> `SWEEP_SUBSET`, or trim the `configs` dict (the LLM/RankGPT reranker is the heaviest).
> Cross-encoder and ColBERT run locally for free; only the judge and generator use Groq.

## Fix #1 — Bigger eval set: RAGAS synthetic test-set generation

Five hand-written factoid questions are too few (and too easy) to trust. RAGAS builds a
knowledge graph over the 16 pages and synthesizes a varied set of questions — single-hop
facts, multi-hop, and more abstract ones — each with a reference answer. We combine the
generated questions with the 5 hand-written anchors into `eval_data_full`.

> The cell is wrapped in `try/except`: synthetic generation is LLM-heavy and the first
> thing to hit a Groq 429. If it fails it falls back to the 5 anchors so the rest still runs.

In [24]:
from ragas.testset import TestsetGenerator

# Build the test set with a SEPARATE Groq model, so the questions aren't shaped by
# the model we're evaluating. (transforms + generation are LLM-heavy -> keep N modest.)
gen_llm = LangchainLLMWrapper(ChatGroq(model="llama-3.3-70b-versatile", temperature=0))
gen_embeddings = LangchainEmbeddingsWrapper(embedding)

N_SYNTHETIC = 8   # raise once your Groq daily-token quota allows

try:
    generator = TestsetGenerator(llm=gen_llm, embedding_model=gen_embeddings)
    synth = generator.generate_with_langchain_docs(
        pages,
        testset_size=N_SYNTHETIC,
        run_config=RunConfig(max_workers=2),   # respect Groq rate limits
    )
    synth_df = synth.to_pandas()
    synthetic = [
        {"question": r["user_input"], "ground_truth": r["reference"]}
        for _, r in synth_df.iterrows()
        if isinstance(r["reference"], str) and r["reference"].strip()
    ]
    print(f"Generated {len(synthetic)} synthetic Q/A pairs")
except Exception as e:
    # Synthetic generation is the most fragile step (Groq quota / 429 / parse errors).
    # Fall back to just the hand-written anchors so the rest of the notebook still runs.
    print("Synthetic generation failed -> using hand-written set only:", repr(e))
    synthetic = []

# Final eval set = 5 hand-written anchors + the synthetic ones
eval_data_full = eval_data + synthetic
print(f"Total eval questions: {len(eval_data_full)}")

Synthetic generation failed -> using hand-written set only: ImportError('rapidfuzz is required for string distance. Please install it using `pip install rapidfuzz`')
Total eval questions: 5


/var/folders/0m/db48z_8d0m5gfh8zkg4vrk100000gn/T/ipykernel_1642/3548129618.py:5: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  gen_llm = LangchainLLMWrapper(ChatGroq(model="llama-3.3-70b-versatile", temperature=0))
/var/folders/0m/db48z_8d0m5gfh8zkg4vrk100000gn/T/ipykernel_1642/3548129618.py:6: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  gen_embeddings = LangchainEmbeddingsWrapper(embedding)


## Fix #2 — Smaller chunks: re-chunk + rebuild the hybrid retriever

Page-level chunks are coarse: a whole page can be only *half* relevant, which caps
`context_precision` no matter how good retrieval is. We re-split the PDF into ~500-char
chunks (80 overlap) and rebuild BM25 + LanceDB + the ensemble on them, with a **wide
candidate pool** (`POOL_K`) so the rerankers have room to work.

In [25]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Re-split the 16 page-chunks into ~500-char chunks (80 overlap). Smaller units mean
# a retrieved chunk is "all relevant" more often -> higher achievable precision, and
# the reranker has finer-grained pieces to score.
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80)
small_pages = splitter.split_documents(pages)
print(f"{len(pages)} page-chunks -> {len(small_pages)} small chunks")

POOL_K = 12  # wide candidate pool so rerankers have real room to work

# Dense retriever over the small chunks, in its own LanceDB table.
small_docsearch = LanceDB.from_documents(
    small_pages, embedding, connection=db,
    table_name="pandas_docs_small", mode="overwrite",
)
small_bm25 = BM25Retriever.from_documents(small_pages); small_bm25.k = POOL_K
small_dense = small_docsearch.as_retriever(search_kwargs={"k": POOL_K})
ensemble_small = EnsembleRetriever(
    retrievers=[small_bm25, small_dense], weights=[0.2, 0.8]
)


def retrieve_pool(query):
    """Wide hybrid candidate pool over the small chunks."""
    return [d.page_content for d in ensemble_small.invoke(query)]


def select_contexts(query, ranker=None, top_k=3):
    """top_k contexts: by hybrid fusion (ranker=None) or after reranking the pool."""
    texts = retrieve_pool(query)
    if ranker is None:
        return texts[:top_k]
    ranked = ranker.rank(query=query, docs=texts)
    return [r.text for r in ranked.top_k(top_k)]


# quick sanity check
print("pool size:", len(retrieve_pool("what is multi-head attention?")))

16 page-chunks -> 96 small chunks
pool size: 18


## Fix #3 — Separate judge model (judge ≠ generator)

In the first evaluation the *same* Groq model wrote the answers **and** graded them, which
inflates `faithfulness` / `answer_relevancy` (a model tends to trust its own output). Here
we keep `openai/gpt-oss-120b` as the answer **generator** but switch the RAGAS **judge** to
`llama-3.3-70b-versatile`, removing the self-grading bias.

In [26]:
# Separate judge: keep openai/gpt-oss-120b as the ANSWER GENERATOR (`llm`),
# but grade with a DIFFERENT model so the RAG isn't marking its own homework.
judge_llm = LangchainLLMWrapper(
    ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
)

# answer_relevancy still needs n=1 on Groq (it defaults to 3 completions per call).
answer_relevancy.strictness = 1

print("Generator: openai/gpt-oss-120b  |  Judge: llama-3.3-70b-versatile")

Generator: openai/gpt-oss-120b  |  Judge: llama-3.3-70b-versatile


/var/folders/0m/db48z_8d0m5gfh8zkg4vrk100000gn/T/ipykernel_1642/1707701370.py:3: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  judge_llm = LangchainLLMWrapper(


## Putting it together: reranker sweep A/B (the real test)

Same questions, same wide pool over the **small chunks**, same generator — only the
**final top-3 selection** changes. We compare the baseline (top-3 by hybrid fusion)
against each reranker, scored by the **separate judge** on all four metrics.
The `delta` columns show each reranker's lift (or drop) versus the baseline.

This is the honest version of the earlier A/B: more questions, cleaner chunks, an
unbiased judge, and three rerankers instead of one — so a positive delta here actually
means something.

In [27]:
import pandas as pd
from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy

sweep_metrics = [context_precision, context_recall, faithfulness, answer_relevancy]
metric_names = [m.name for m in sweep_metrics]

# Trim to control Groq usage; bump once your daily token quota allows.
SWEEP_SUBSET = eval_data_full[:8]

# Rerankers to compare (all reuse the models built earlier in the notebook).
# Drop entries if you hit a 429 -- the LLM (RankGPT) one is the most token-hungry.
configs = {
    "baseline (top-3 hybrid)": None,
    "cross-encoder":           ce_ranker,
    "ColBERT":                 colbert_reranker,
    "LLM (RankGPT)":           llm_ranker,
}


def build_eval_dataset(ranker):
    """Wide pool over small chunks -> keep top-3 (by fusion or rerank) -> generate."""
    samples = []
    for row in SWEEP_SUBSET:
        q = row["question"]
        texts = select_contexts(q, ranker=ranker, top_k=3)
        msgs = rag_prompt.format_messages(context="\n\n".join(texts), question=q)
        answer = llm.invoke(msgs).content            # generator = openai/gpt-oss-120b
        samples.append({
            "user_input": q, "retrieved_contexts": texts,
            "response": answer, "reference": row["ground_truth"],
        })
    return EvaluationDataset.from_list(samples)


cfg = RunConfig(max_workers=1)  # gentle on Groq rate limits
scores = {}
for name, ranker in configs.items():
    ev = evaluate(
        dataset=build_eval_dataset(ranker), metrics=sweep_metrics,
        llm=judge_llm,                     # <-- separate judge model (NOT the generator)
        embeddings=evaluator_embeddings, run_config=cfg,
    )
    df = ev.to_pandas()
    scores[name] = {n: df[n].mean() for n in metric_names}
    print(f"done: {name}")

sweep = pd.DataFrame(scores)
for name in configs:
    if name != "baseline (top-3 hybrid)":
        sweep[f"delta {name}"] = sweep[name] - sweep["baseline (top-3 hybrid)"]
sweep.round(4)

/var/folders/0m/db48z_8d0m5gfh8zkg4vrk100000gn/T/ipykernel_1642/196626069.py:2: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy
/var/folders/0m/db48z_8d0m5gfh8zkg4vrk100000gn/T/ipykernel_1642/196626069.py:2: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy
/var/folders/0m/db48z_8d0m5gfh8zkg4vrk100000gn/T/ipykernel_1642/196626069.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.met

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01k5hjn0e6fyxbf0wfmz4aha4n` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199120, Requested 1689. Please try again in 5m49.488s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}